# Backtesting ML Classification-Based

![](src/res_classification.png)

## Load the model

In [2]:
import pickle

In [3]:
with open('models/model_dt_classification.pkl', 'rb') as f:
    model_dt = pickle.load(f)

In [4]:
model_dt

DecisionTreeClassifier(max_depth=15)

## Load the data

In [5]:
import pandas as pd

df = pd.read_excel('data/Microsoft_LinkedIn_Processed.xlsx', index_col=0, parse_dates=['Date'])
df

,Close,High,Low,Open,Volume,change_tomorrow,change_tomorrow_direction
Date,,,,,,,
2016-12-08,61.009998,61.580002,60.840000,61.299999,21220800,1.549141,UP
2016-12-09,61.970001,61.990002,61.130001,61.180000,27349400,0.321694,UP
2016-12-12,62.169998,62.299999,61.720001,61.820000,20198100,1.286125,UP
2016-12-13,62.980000,63.419998,62.240002,62.500000,35718900,-0.478620,DOWN
2016-12-14,62.680000,63.450001,62.529999,63.000000,30352700,-0.159793,DOWN
...,...,...,...,...,...,...,...
2026-05-14,409.429993,411.839996,400.880005,404.480011,27077500,2.960282,UP
2026-05-15,421.920013,428.170013,412.910004,414.269989,50771200,0.382489,UP
2026-05-18,423.540009,425.119995,415.609985,416.619995,32564100,-1.466148,DOWN


## Backtesting.py Library

### Create your Strategy Class

In [6]:
import sys
!{sys.executable} -m pip install backtesting

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [7]:
from backtesting import Backtest,Strategy

In [8]:
df_explanatory= df.drop(columns= ['change_tomorrow', 'change_tomorrow_direction'])

In [9]:
model_dt.predict(X=df_explanatory)

array(['UP', 'UP', 'UP', ..., 'DOWN', 'DOWN', 'DOWN'], dtype=object)

In [10]:
explanatory_today= df_explanatory.iloc[[-1],:]

In [11]:
forecast_tomorrow= model_dt.predict(explanatory_today)[0]

#### Simulate the prediction for the last observation

#### Write the prediction process in the Strategy class

In [12]:
class ClassificationUP(Strategy):
    def init(self):
        self.model = model_dt

    def next(self):
        explanatory_today = df_explanatory.iloc[[-1],:]
        forecast_tomorrow = model_dt.predict(explanatory_today)[0]
        
        # conditions to sell or buy

#### Calculate Purchase Recommendation

##### Buy if it goes up

In [13]:
list_buy_sell = []

In [14]:
for tomorrow_direction in df.change_tomorrow_direction:
  if tomorrow_direction == 'UP':
    list_buy_sell.append(1)
  elif tomorrow_direction == 'DOWN':
    list_buy_sell.append(-1)

In [15]:
df_buy_sell = list_buy_sell

In [16]:
print(df.columns.tolist())

['Close', 'High', 'Low', 'Open', 'Volume', 'change_tomorrow', 'change_tomorrow_direction']


In [17]:
df['buy_sell'] = df['change_tomorrow_direction'].map({'UP': 1, 'DOWN': -1})

In [18]:
df[['change_tomorrow_direction','buy_sell']].head(10)

,change_tomorrow_direction,buy_sell
Date,,
2016-12-08,UP,1
2016-12-09,UP,1
2016-12-12,UP,1
2016-12-13,DOWN,-1
2016-12-14,DOWN,-1
2016-12-15,DOWN,-1
2016-12-16,UP,1
2016-12-19,DOWN,-1
2016-12-20,DOWN,-1


##### Buy if it goes and sell if down

> You can only sell if you have already bought

In [19]:
list_buy_sell = []
already_bought = False

In [20]:
for tomorrow_direction in df['change_tomorrow_direction']:
  if tomorrow_direction == 'UP'and already_bought == False:
    list_buy_sell.append(1)
    already_bought = True
  elif tomorrow_direction == 'DOWN' and already_bought == True:
    list_buy_sell.append(-1)
    already_bought = False 
  else:
    list_buy_sell.append(0)

In [21]:
print(len(list_buy_sell))
print(len(df))

2374
2374


In [22]:
df['buy_sell_track']= list_buy_sell

In [23]:
df[['change_tomorrow_direction','buy_sell','buy_sell_track']].head(10)

,change_tomorrow_direction,buy_sell,buy_sell_track
Date,,,
2016-12-08,UP,1,1
2016-12-09,UP,1,0
2016-12-12,UP,1,0
2016-12-13,DOWN,-1,-1
2016-12-14,DOWN,-1,0
2016-12-15,DOWN,-1,0
2016-12-16,UP,1,1
2016-12-19,DOWN,-1,-1
2016-12-20,DOWN,-1,0


#### Add conditions to the strategy

In [29]:
class ClassificationUP(Strategy):
    def init(self):
        self.model = model_dt
        self.already_bought = False

    def next(self):
        explanatory_today =self.data.df.iloc[[-1],:]
        forecast_tomorrow = model_dt.predict(explanatory_today)[0]
        
        # conditions to sell or buy

        if forecast_tomorrow == 'UP'and self.already_bought == False:
            self.buy()
            already_bought = True
        elif tomorrow_direction == 'DOWN' and self.already_bought == True:
            self.sell()
            already_bought = False 
        else:
            pass

### Define initial conditions

In [25]:
import backtesting
print(backtesting.__version__)

0.3.3


In [33]:
import sys
!{sys.executable} -m pip install backtesting==0.3.3 --force-reinstall

Defaulting to user installation because normal site-packages is not writeable
  Using cached Backtesting-0.3.3-py3-none-any.whl
  Using cached bokeh-3.4.3-py3-none-any.whl (7.0 MB)
  Using cached numpy-2.0.2-cp39-cp39-macosx_14_0_arm64.whl (5.3 MB)
  Using cached pandas-2.3.3-cp39-cp39-macosx_11_0_arm64.whl (10.8 MB)
  Using cached pyyaml-6.0.3-cp39-cp39-macosx_11_0_arm64.whl (174 kB)
  Using cached packaging-26.2-py3-none-any.whl (100 kB)
  Using cached tornado-6.5.5-cp39-abi3-macosx_10_9_universal2.whl (445 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
  Using cached pillow-11.3.0-cp39-cp39-macosx_11_0_arm64.whl (4.7 MB)
  Using cached contourpy-1.3.0-cp39-cp39-macosx_11_0_arm64.whl (249 kB)
  Using cached xyzservices-2026.3.0-py3-none-any.whl (94 kB)
  Using cached markupsafe-3.0.3-cp39-cp39-macosx_11_0_arm64.whl (12 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl (349 kB)
  Using cached pytz-2026.2-py2.py3-none-any.whl (510 kB)
  Using cached python_dateutil-2.9.0

In [34]:
from backtesting import Backtest, Strategy

bt = Backtest(df_explanatory, ClassificationUP,
              cash=10000, commission=.002, exclusive_orders=True)

### Run backtesting

In [37]:
results=bt.run()

### Interpret backtesting results

In [38]:
results.to_frame(name='Values').loc[:'Return [%]']

,Values
Start,2016-12-08 00:00:00
End,2026-05-20 00:00:00
Duration,3450 days 00:00:00
Exposure Time [%],99.915754
Equity Final [$],4580.131935
Equity Peak [$],11845.333989
Return [%],-54.198681


## Practice to master the knowledge

Work on the challenge with another dataset:

1. Learn the <a>mental models</a> to solve the challenge faster.
2. Complete the <a href="03C_Backtesting ML Classification-Based.ipynb">notebook</a>.